# Simple WBF Ensemble: RT-DETR-L + YOLO11m
## Source-Based

Notebook ini **tidak melakukan training detector**.

Notebook hanya:

1. Memuat checkpoint yang sudah ada.
2. Membuat common filtered COCO ground truth untuk validation dan test.
3. Menjalankan raw inference pada seluruh gambar, termasuk dot-prefixed images.
4. Menghitung skor individual dengan COCOeval yang sama.
5. Menyetel parameter WBF sederhana pada validation.
6. Mengunci parameter terbaik dan mengevaluasi ensemble satu kali pada test.

Class mapping:

- Ultralytics: `0 = coral`, `1 = fish`
- Faster R-CNN: `0 = background`, `1 = coral`, `2 = fish`
- COCOeval: `1 = coral`, `2 = fish`

In [1]:
# CELL 1 — INSTALL
!pip -q install ultralytics==8.4.102 pycocotools ensemble-boxes pandas pillow tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 70.7 MB/s eta 0:00:00


In [2]:
# CELL 2 — MOUNT DRIVE
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# CELL 3 — CONFIGURATION
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import contextlib
import gc
import hashlib
import io
import json
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision

from PIL import Image
from IPython.display import display
from tqdm.auto import tqdm

from ultralytics import YOLO, RTDETR
from ensemble_boxes import weighted_boxes_fusion
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F_tv
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

SPLIT_NAME = 'source_based'
COMBINATION_NAME = 'rtdetr_l_yolo11m'
MODEL_NAMES = ['rtdetr_l', 'yolo11m']

PROJECT_ROOT = Path('/content/drive/MyDrive/au-fish/dataset_v2')
DATASET_ROOT = PROJECT_ROOT / SPLIT_NAME
VAL_DIR = DATASET_ROOT / "val"
TEST_DIR = DATASET_ROOT / "test"

BASE_OUTPUT_DIR = (
    DATASET_ROOT
    / "experiment_logs"
    / "simple_wbf_fullset_dedup_v1"
)

OUTPUT_DIR = BASE_OUTPUT_DIR / COMBINATION_NAME

# Shared across all three combinations in this split.
GT_DIR = BASE_OUTPUT_DIR / "shared_ground_truth"
RAW_DIR = BASE_OUTPUT_DIR / "shared_raw_predictions"

RESULT_DIR = OUTPUT_DIR / "results"

# RESET_OUTPUT only clears this combination's result folder.
# Shared raw predictions remain available for other combinations.
RESET_OUTPUT = False
REUSE_RAW_PREDICTIONS = True

MODEL_SPECS = {'rtdetr_l': {'family': 'rtdetr', 'path': '/content/drive/MyDrive/au-fish/dataset_v2/source_based/runs/rtdetr_l_source_based_fullset_dedup_v1/weights/best.pt', 'batch': 4}, 'yolo11m': {'family': 'yolo', 'path': '/content/drive/MyDrive/au-fish/dataset_v2/source_based/runs/yolo11m_source_based_fullset_dedup_v1/weights/best.pt', 'batch': 16}}
EXPECTED_COUNTS = {'validation': {'images': 232, 'annotations': 1857, 'hidden_images': 9}, 'test': {'images': 231, 'annotations': 1820, 'hidden_images': 11}}

CLASS_NAMES = ["coral", "fish"]
KEEP_CLASSES = set(CLASS_NAMES)
MIN_BOX_SIZE = 5
MIN_BOX_AREA = 25

DEVICE_ULTRA = 0 if torch.cuda.is_available() else "cpu"
DEVICE_TORCH = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

IMGSZ = 640
RAW_CONF = 0.001
RAW_IOU = 0.70
RAW_MAX_DET = 300

# Tuning sederhana dan cukup ringan.
IOU_GRID = [0.45, 0.50, 0.55, 0.60]
SKIP_GRID = [0.001, 0.01, 0.03]
CONF_TYPES = ["avg", "box_and_model_avg", "max"]
WEIGHT_GRID = [0.75, 1.00, 1.25, 1.50]

print("Split       :", SPLIT_NAME)
print("Combination :", COMBINATION_NAME)
print("Models      :", MODEL_NAMES)
print("Output      :", OUTPUT_DIR)
print("Device      :", DEVICE_TORCH)
print("No detector training will be performed.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Split       : source_based
Combination : rtdetr_l_yolo11m
Models      : ['rtdetr_l', 'yolo11m']
Output      : /content/drive/MyDrive/au-fish/dataset_v2/source_based/experiment_logs/simple_wbf_fullset_dedup_v1/rtdetr_l_yolo11m
Device      : cuda
No detector training will be performed.


In [4]:
# CELL 4 — BASIC HELPERS AND OUTPUT SETUP
def save_json(data, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )


def load_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(chunk_size),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


if RESET_OUTPUT and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

for directory in [
    OUTPUT_DIR,
    GT_DIR,
    RAW_DIR,
    RESULT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

checkpoint_rows = []

for model_name, spec in MODEL_SPECS.items():
    checkpoint_path = Path(spec["path"])

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Checkpoint tidak ditemukan: {checkpoint_path}"
        )

    checkpoint_rows.append({
        "model": model_name,
        "family": spec["family"],
        "path": str(checkpoint_path),
        "sha256": sha256_file(checkpoint_path),
        "size_bytes": checkpoint_path.stat().st_size,
    })

checkpoint_df = pd.DataFrame(checkpoint_rows)
checkpoint_df.to_csv(
    OUTPUT_DIR / "checkpoint_manifest.csv",
    index=False,
)

display(checkpoint_df)

,model,family,path,sha256,size_bytes
0,rtdetr_l,rtdetr,/content/drive/MyDrive/au-fish/dataset_v2/sour...,2a8873219fcdc498498cd8c1c639871df1f7c34a185a15...,66220541
1,yolo11m,yolo,/content/drive/MyDrive/au-fish/dataset_v2/sour...,b65bae9454bb36387bf32d8591d676dfdddaf3905f4a48...,40501292


In [5]:
# ============================================================
# SAFE INFERENCE BATCH
# ============================================================
# RT-DETR-L membutuhkan VRAM jauh lebih besar.
if "rtdetr_l" in MODEL_SPECS:
    MODEL_SPECS["rtdetr_l"]["batch"] = 1

# YOLO11m masih aman menggunakan batch kecil.
if "yolo11m" in MODEL_SPECS:
    MODEL_SPECS["yolo11m"]["batch"] = 4

In [6]:
# CELL 5 — BUILD COMMON FILTERED COCO GT
def build_filtered_gt(split_dir, split_label, output_path):
    split_dir = Path(split_dir)
    source = load_json(
        split_dir / "annotations.json"
    )

    image_map = {
        int(image["id"]): image
        for image in source["images"]
    }

    category_name = {
        int(category["id"]): category["name"]
        for category in source["categories"]
    }

    final_category = {
        "coral": 1,
        "fish": 2,
    }

    annotations = []
    seen = set()
    class_counts = {
        "coral": 0,
        "fish": 0,
    }

    annotation_id = 1

    for annotation in source["annotations"]:
        image_id = int(annotation["image_id"])

        if image_id not in image_map:
            continue

        name = category_name.get(
            int(annotation["category_id"])
        )

        if name not in KEEP_CLASSES:
            continue

        bbox = annotation.get("bbox")

        if (
            not isinstance(bbox, (list, tuple))
            or len(bbox) != 4
        ):
            continue

        x, y, width, height = map(float, bbox)
        image = image_map[image_id]

        image_width = float(image["width"])
        image_height = float(image["height"])

        x1 = max(0.0, x)
        y1 = max(0.0, y)
        x2 = min(image_width, x + width)
        y2 = min(image_height, y + height)

        clipped_width = x2 - x1
        clipped_height = y2 - y1

        if (
            clipped_width < MIN_BOX_SIZE
            or clipped_height < MIN_BOX_SIZE
            or clipped_width * clipped_height < MIN_BOX_AREA
        ):
            continue

        category_id = final_category[name]

        signature = (
            image_id,
            category_id,
            round(x1, 6),
            round(y1, 6),
            round(clipped_width, 6),
            round(clipped_height, 6),
            int(annotation.get("iscrowd", 0)),
        )

        if signature in seen:
            continue

        seen.add(signature)

        annotations.append({
            "id": annotation_id,
            "image_id": image_id,
            "category_id": category_id,
            "bbox": [
                x1,
                y1,
                clipped_width,
                clipped_height,
            ],
            "area": clipped_width * clipped_height,
            "iscrowd": int(
                annotation.get("iscrowd", 0)
            ),
        })

        class_counts[name] += 1
        annotation_id += 1

    images = source["images"]

    missing_images = [
        image["file_name"]
        for image in images
        if not (
            split_dir
            / "images"
            / image["file_name"]
        ).exists()
    ]

    if missing_images:
        raise FileNotFoundError(
            f"{split_label}: missing images: "
            f"{missing_images[:5]}"
        )

    hidden_images = sum(
        Path(image["file_name"]).name.startswith(".")
        for image in images
    )

    actual = {
        "images": len(images),
        "annotations": len(annotations),
        "hidden_images": hidden_images,
        "coral": class_counts["coral"],
        "fish": class_counts["fish"],
    }

    expected = EXPECTED_COUNTS[split_label]

    for key in [
        "images",
        "annotations",
        "hidden_images",
    ]:
        if actual[key] != expected[key]:
            raise RuntimeError(
                f"{split_label} {key} mismatch: "
                f"{actual[key]} != {expected[key]}"
            )

    filtered = {
        "info": source.get("info", {}),
        "licenses": source.get("licenses", []),
        "images": images,
        "annotations": annotations,
        "categories": [
            {
                "id": 1,
                "name": "coral",
                "supercategory": "object",
            },
            {
                "id": 2,
                "name": "fish",
                "supercategory": "object",
            },
        ],
    }

    save_json(filtered, output_path)
    return filtered, actual


VAL_GT_PATH = GT_DIR / "val_gt_filtered.json"
TEST_GT_PATH = GT_DIR / "test_gt_filtered.json"

val_gt, val_audit = build_filtered_gt(
    VAL_DIR,
    "validation",
    VAL_GT_PATH,
)

test_gt, test_audit = build_filtered_gt(
    TEST_DIR,
    "test",
    TEST_GT_PATH,
)

VAL_COCO = COCO(str(VAL_GT_PATH))
TEST_COCO = COCO(str(TEST_GT_PATH))

gt_audit_df = pd.DataFrame([
    {"split": "validation", **val_audit},
    {"split": "test", **test_audit},
])

gt_audit_df.to_csv(
    OUTPUT_DIR / "ground_truth_audit.csv",
    index=False,
)

display(gt_audit_df)

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


,split,images,annotations,hidden_images,coral,fish
0,validation,232,1857,9,288,1569
1,test,231,1820,11,437,1383


In [7]:
# CELL 6 — PREDICTION HELPERS
def blank_record(image_info):
    return {
        "image_id": int(image_info["id"]),
        "file_name": image_info["file_name"],
        "width": int(image_info["width"]),
        "height": int(image_info["height"]),
        "boxes": [],
        "scores": [],
        "labels": [],
    }


def validate_prediction_coverage(
    predictions,
    gt_path,
    model_name,
    split_label,
):
    gt_ids = {
        str(int(image["id"]))
        for image in load_json(gt_path)["images"]
    }

    prediction_ids = set(predictions.keys())

    if prediction_ids != gt_ids:
        raise RuntimeError(
            f"{model_name}/{split_label}: coverage mismatch. "
            f"missing={len(gt_ids - prediction_ids)}, "
            f"extra={len(prediction_ids - gt_ids)}"
        )


def clear_gpu_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def predict_ultralytics(
    model_name,
    spec,
    split_dir,
    gt_path,
):
    image_infos = load_json(gt_path)["images"]

    image_paths = [
        str(
            Path(split_dir)
            / "images"
            / image_info["file_name"]
        )
        for image_info in image_infos
    ]

    missing_paths = [
        image_path
        for image_path in image_paths
        if not Path(image_path).exists()
    ]

    if missing_paths:
        raise FileNotFoundError(
            f"{model_name}: ditemukan "
            f"{len(missing_paths)} image yang hilang. "
            f"Contoh: {missing_paths[:5]}"
        )

    clear_gpu_memory()

    model = (
        RTDETR(str(spec["path"]))
        if spec["family"] == "rtdetr"
        else YOLO(str(spec["path"]))
    )

    # RT-DETR selalu batch 1 agar aman.
    if spec["family"] == "rtdetr":
        inference_batch = 1
    else:
        inference_batch = min(
            int(spec.get("batch", 4)),
            4,
        )

    predictions = {}
    processed = 0

    try:
        progress = tqdm(
            total=len(image_infos),
            desc=f"Predict {model_name}",
        )

        for start_index in range(
            0,
            len(image_paths),
            inference_batch,
        ):
            end_index = min(
                start_index + inference_batch,
                len(image_paths),
            )

            batch_paths = image_paths[
                start_index:end_index
            ]

            batch_infos = image_infos[
                start_index:end_index
            ]

            try:
                # stream=False hanya menyimpan satu batch kecil
                # dalam memori.
                batch_results = model.predict(
                    source=batch_paths,
                    imgsz=IMGSZ,
                    conf=RAW_CONF,
                    iou=RAW_IOU,
                    max_det=RAW_MAX_DET,
                    batch=len(batch_paths),
                    device=DEVICE_ULTRA,
                    stream=False,
                    verbose=False,
                    augment=False,
                    half=torch.cuda.is_available(),
                )

            except torch.OutOfMemoryError:
                # Fallback otomatis menjadi satu image per inference.
                clear_gpu_memory()

                print(
                    f"\n⚠️ OOM pada {model_name}, "
                    "retry menggunakan batch=1."
                )

                batch_results = []

                for image_path in batch_paths:
                    single_result = model.predict(
                        source=[image_path],
                        imgsz=IMGSZ,
                        conf=RAW_CONF,
                        iou=RAW_IOU,
                        max_det=RAW_MAX_DET,
                        batch=1,
                        device=DEVICE_ULTRA,
                        stream=False,
                        verbose=False,
                        augment=False,
                        half=torch.cuda.is_available(),
                    )

                    if len(single_result) != 1:
                        raise RuntimeError(
                            f"{model_name}: single-image "
                            "inference tidak menghasilkan "
                            "tepat satu output."
                        )

                    batch_results.append(
                        single_result[0]
                    )

                    del single_result
                    clear_gpu_memory()

            if len(batch_results) != len(batch_infos):
                raise RuntimeError(
                    f"{model_name}: jumlah output batch "
                    f"{len(batch_results)} tidak sama "
                    f"dengan input {len(batch_infos)}."
                )

            for image_info, result in zip(
                batch_infos,
                batch_results,
            ):
                # Mapping memakai urutan input.
                # Jangan memakai result.path karena Ultralytics
                # dapat mengubahnya menjadi image0.jpg.
                record = blank_record(image_info)

                if (
                    result.boxes is not None
                    and len(result.boxes) > 0
                ):
                    boxes = (
                        result.boxes.xyxy
                        .detach()
                        .cpu()
                        .numpy()
                    )

                    scores = (
                        result.boxes.conf
                        .detach()
                        .cpu()
                        .numpy()
                    )

                    labels = (
                        result.boxes.cls
                        .detach()
                        .cpu()
                        .numpy()
                        .astype(int)
                    )

                    for box, score, label in zip(
                        boxes,
                        scores,
                        labels,
                    ):
                        if int(label) not in [0, 1]:
                            continue

                        record["boxes"].append([
                            float(value)
                            for value in box
                        ])

                        record["scores"].append(
                            float(score)
                        )

                        record["labels"].append(
                            int(label)
                        )

                image_id = str(
                    int(record["image_id"])
                )

                if image_id in predictions:
                    raise RuntimeError(
                        f"{model_name}: duplicate prediction "
                        f"untuk image_id={image_id}"
                    )

                predictions[image_id] = record
                processed += 1
                progress.update(1)

            del batch_results

            # Membersihkan cache secara berkala.
            if processed % 10 == 0:
                clear_gpu_memory()

        progress.close()

    finally:
        del model
        clear_gpu_memory()

    if processed != len(image_infos):
        raise RuntimeError(
            f"{model_name}: processed={processed}, "
            f"expected={len(image_infos)}"
        )

    if len(predictions) != len(image_infos):
        raise RuntimeError(
            f"{model_name}: prediction records="
            f"{len(predictions)}, "
            f"expected={len(image_infos)}"
        )

    return predictions


def load_frcnn_state(path):
    try:
        state = torch.load(
            path,
            map_location="cpu",
            weights_only=True,
        )
    except TypeError:
        state = torch.load(
            path,
            map_location="cpu",
        )

    if isinstance(state, dict):
        for key in [
            "model_state_dict",
            "state_dict",
            "model",
        ]:
            if (
                key in state
                and isinstance(state[key], dict)
            ):
                state = state[key]
                break

    return {
        (
            key[7:]
            if key.startswith("module.")
            else key
        ): value
        for key, value in state.items()
    }


def build_faster_rcnn(checkpoint_path):
    model = fasterrcnn_resnet50_fpn(
        weights=None,
        weights_backbone=None,
    )

    in_features = (
        model.roi_heads
        .box_predictor
        .cls_score
        .in_features
    )

    model.roi_heads.box_predictor = (
        FastRCNNPredictor(
            in_features,
            3,
        )
    )

    model.load_state_dict(
        load_frcnn_state(checkpoint_path),
        strict=True,
    )

    return model.to(
        DEVICE_TORCH
    ).eval()


class FRCNNInferenceDataset(Dataset):
    def __init__(self, split_dir, gt_path):
        self.image_dir = (
            Path(split_dir) / "images"
        )
        self.image_infos = load_json(
            gt_path
        )["images"]

    def __len__(self):
        return len(self.image_infos)

    def __getitem__(self, index):
        image_info = self.image_infos[index]
        image_path = (
            self.image_dir
            / image_info["file_name"]
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        return (
            F_tv.to_tensor(image),
            image_info,
        )


def collate_frcnn(batch):
    return tuple(zip(*batch))


def predict_faster_rcnn(
    model_name,
    spec,
    split_dir,
    gt_path,
):
    dataset = FRCNNInferenceDataset(
        split_dir,
        gt_path,
    )

    loader = DataLoader(
        dataset,
        batch_size=int(spec["batch"]),
        shuffle=False,
        num_workers=0,
        collate_fn=collate_frcnn,
        pin_memory=torch.cuda.is_available(),
    )

    model = build_faster_rcnn(
        spec["path"]
    )

    predictions = {}

    with torch.inference_mode():
        for images, image_infos in tqdm(
            loader,
            desc=f"Predict {model_name}",
        ):
            outputs = model([
                image.to(DEVICE_TORCH)
                for image in images
            ])

            for image_info, output in zip(
                image_infos,
                outputs,
            ):
                record = blank_record(image_info)

                boxes = (
                    output["boxes"]
                    .detach()
                    .cpu()
                    .numpy()
                )
                scores = (
                    output["scores"]
                    .detach()
                    .cpu()
                    .numpy()
                )
                labels = (
                    output["labels"]
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(int)
                )

                for box, score, label in zip(
                    boxes,
                    scores,
                    labels,
                ):
                    if (
                        float(score) < RAW_CONF
                        or int(label) not in [1, 2]
                    ):
                        continue

                    record["boxes"].append([
                        float(value)
                        for value in box
                    ])
                    record["scores"].append(
                        float(score)
                    )
                    record["labels"].append(
                        int(label - 1)
                    )

                predictions[
                    str(record["image_id"])
                ] = record

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return predictions


def get_predictions(
    model_name,
    split_label,
    split_dir,
    gt_path,
):
    output_path = (
        RAW_DIR
        / f"{split_label}_{model_name}.json"
    )

    if (
        REUSE_RAW_PREDICTIONS
        and output_path.exists()
    ):
        predictions = load_json(output_path)
        validate_prediction_coverage(
            predictions,
            gt_path,
            model_name,
            split_label,
        )

        print(
            f"Reused: {output_path.name}"
        )
        return predictions

    spec = MODEL_SPECS[model_name]

    if spec["family"] in ["yolo", "rtdetr"]:
        predictions = predict_ultralytics(
            model_name,
            spec,
            split_dir,
            gt_path,
        )
    else:
        predictions = predict_faster_rcnn(
            model_name,
            spec,
            split_dir,
            gt_path,
        )

    validate_prediction_coverage(
        predictions,
        gt_path,
        model_name,
        split_label,
    )

    save_json(predictions, output_path)
    print("Saved:", output_path)

    return predictions

In [8]:
# ============================================================
# GPU CLEANUP BEFORE INFERENCE
# ============================================================
import gc
import torch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

    free_memory, total_memory = torch.cuda.mem_get_info()

    print(
        "Free CUDA memory :",
        f"{free_memory / 1024**3:.2f} GB",
    )

    print(
        "Total CUDA memory:",
        f"{total_memory / 1024**3:.2f} GB",
    )

Free CUDA memory : 21.84 GB
Total CUDA memory: 22.03 GB


In [9]:
# CELL 7 — RUN OR REUSE RAW INFERENCE
validation_predictions = {}
test_predictions = {}

for model_name in MODEL_NAMES:
    print("\n" + "=" * 80)
    print("VALIDATION:", model_name)
    print("=" * 80)

    validation_predictions[model_name] = (
        get_predictions(
            model_name,
            "validation",
            VAL_DIR,
            VAL_GT_PATH,
        )
    )

    print("\n" + "=" * 80)
    print("TEST:", model_name)
    print("=" * 80)

    test_predictions[model_name] = (
        get_predictions(
            model_name,
            "test",
            TEST_DIR,
            TEST_GT_PATH,
        )
    )

print("✅ Raw inference complete.")


VALIDATION: rtdetr_l
Reused: validation_rtdetr_l.json

TEST: rtdetr_l
Reused: test_rtdetr_l.json

VALIDATION: yolo11m
Reused: validation_yolo11m.json

TEST: yolo11m
Reused: test_yolo11m.json
✅ Raw inference complete.


In [10]:
# CELL 8 — COMMON COCOEVAL
def predictions_to_coco(predictions):
    coco_rows = []

    for record in predictions.values():
        image_id = int(record["image_id"])

        for box, score, label in zip(
            record["boxes"],
            record["scores"],
            record["labels"],
        ):
            x1, y1, x2, y2 = map(float, box)
            width = max(0.0, x2 - x1)
            height = max(0.0, y2 - y1)

            if width <= 0 or height <= 0:
                continue

            coco_rows.append({
                "image_id": image_id,
                "category_id": int(label) + 1,
                "bbox": [
                    x1,
                    y1,
                    width,
                    height,
                ],
                "score": float(score),
            })

    return coco_rows


def evaluate_coco(
    coco_gt,
    predictions,
    label,
):
    coco_rows = predictions_to_coco(
        predictions
    )

    if not coco_rows:
        return {
            "name": label,
            "mAP50_95": 0.0,
            "mAP50": 0.0,
            "mAP75": 0.0,
            "AP_coral": 0.0,
            "AP_fish": 0.0,
            "worst_class_AP": 0.0,
            "detections": 0,
        }

    with contextlib.redirect_stdout(io.StringIO()):
        coco_dt = coco_gt.loadRes(coco_rows)
        evaluator = COCOeval(
            coco_gt,
            coco_dt,
            "bbox",
        )
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

    stats = evaluator.stats

    class_ap = {}

    for category_id, class_name in [
        (1, "coral"),
        (2, "fish"),
    ]:
        with contextlib.redirect_stdout(
            io.StringIO()
        ):
            class_evaluator = COCOeval(
                coco_gt,
                coco_dt,
                "bbox",
            )
            class_evaluator.params.catIds = [
                category_id
            ]
            class_evaluator.evaluate()
            class_evaluator.accumulate()
            class_evaluator.summarize()

        class_ap[class_name] = float(
            class_evaluator.stats[0]
        )

    return {
        "name": label,
        "mAP50_95": float(stats[0]),
        "mAP50": float(stats[1]),
        "mAP75": float(stats[2]),
        "AP_coral": class_ap["coral"],
        "AP_fish": class_ap["fish"],
        "worst_class_AP": min(
            class_ap["coral"],
            class_ap["fish"],
        ),
        "detections": len(coco_rows),
    }


individual_rows = []

for split_label, coco_gt, prediction_sets in [
    (
        "validation",
        VAL_COCO,
        validation_predictions,
    ),
    (
        "test",
        TEST_COCO,
        test_predictions,
    ),
]:
    for model_name, predictions in (
        prediction_sets.items()
    ):
        row = evaluate_coco(
            coco_gt,
            predictions,
            model_name,
        )
        row["split"] = split_label
        individual_rows.append(row)

individual_df = pd.DataFrame(
    individual_rows
)

individual_df.to_csv(
    RESULT_DIR / "individual_results.csv",
    index=False,
)

display(
    individual_df.sort_values(
        ["split", "mAP50_95"],
        ascending=[True, False],
    )
)

,name,mAP50_95,mAP50,mAP75,AP_coral,AP_fish,worst_class_AP,detections,split
2,rtdetr_l,0.227832,0.407838,0.215314,0.258369,0.197296,0.197296,69300,test
3,yolo11m,0.207680,0.350498,0.208917,0.242860,0.172501,0.172501,14484,test
0,rtdetr_l,0.293047,0.459558,0.308313,0.389580,0.196514,0.196514,69600,validation
1,yolo11m,0.242929,0.398255,0.250530,0.340161,0.145698,0.145698,11506,validation


In [11]:
# CELL 9 — WBF HELPERS
def fuse_one_image(
    records,
    weights,
    iou_thr,
    skip_box_thr,
    conf_type,
):
    width = float(records[0]["width"])
    height = float(records[0]["height"])

    boxes_list = []
    scores_list = []
    labels_list = []

    for record in records:
        normalized_boxes = []

        for box in record["boxes"]:
            x1, y1, x2, y2 = map(float, box)

            normalized_boxes.append([
                min(max(x1 / width, 0.0), 1.0),
                min(max(y1 / height, 0.0), 1.0),
                min(max(x2 / width, 0.0), 1.0),
                min(max(y2 / height, 0.0), 1.0),
            ])

        boxes_list.append(normalized_boxes)
        scores_list.append([
            float(score)
            for score in record["scores"]
        ])
        labels_list.append([
            int(label)
            for label in record["labels"]
        ])

    if not any(len(boxes) for boxes in boxes_list):
        fused_record = blank_record(records[0])
        return fused_record

    fused_boxes, fused_scores, fused_labels = (
        weighted_boxes_fusion(
            boxes_list,
            scores_list,
            labels_list,
            weights=weights,
            iou_thr=float(iou_thr),
            skip_box_thr=float(skip_box_thr),
            conf_type=conf_type,
            allows_overflow=False,
        )
    )

    fused_record = blank_record(records[0])

    for box, score, label in zip(
        fused_boxes,
        fused_scores,
        fused_labels,
    ):
        x1, y1, x2, y2 = box

        fused_record["boxes"].append([
            float(x1 * width),
            float(y1 * height),
            float(x2 * width),
            float(y2 * height),
        ])
        fused_record["scores"].append(
            float(score)
        )
        fused_record["labels"].append(
            int(label)
        )

    return fused_record


def fuse_prediction_sets(
    prediction_sets,
    model_names,
    weights,
    iou_thr,
    skip_box_thr,
    conf_type,
):
    reference_ids = set(
        prediction_sets[
            model_names[0]
        ].keys()
    )

    for model_name in model_names[1:]:
        current_ids = set(
            prediction_sets[
                model_name
            ].keys()
        )

        if current_ids != reference_ids:
            raise RuntimeError(
                "Prediction image IDs differ "
                "between models."
            )

    fused = {}

    for image_id in sorted(
        reference_ids,
        key=int,
    ):
        records = [
            prediction_sets[
                model_name
            ][image_id]
            for model_name in model_names
        ]

        fused[image_id] = fuse_one_image(
            records,
            weights,
            iou_thr,
            skip_box_thr,
            conf_type,
        )

    return fused


def candidate_key(row):
    return (
        row["mAP50_95"],
        row["worst_class_AP"],
        row["mAP50"],
    )

In [12]:
# ============================================================
# PATCH — BLANK RECORD SUPPORTS COCO AND PREDICTION RECORDS
# ============================================================
def blank_record(source_record):
    """
    Mendukung dua format:

    1. COCO image info:
       {"id", "file_name", "width", "height"}

    2. Prediction record:
       {"image_id", "file_name", "width", "height", ...}
    """
    if "id" in source_record:
        image_id = int(source_record["id"])
    elif "image_id" in source_record:
        image_id = int(source_record["image_id"])
    else:
        raise KeyError(
            "Record tidak memiliki key 'id' maupun 'image_id'. "
            f"Keys tersedia: {list(source_record.keys())}"
        )

    required_fields = [
        "file_name",
        "width",
        "height",
    ]

    missing_fields = [
        field
        for field in required_fields
        if field not in source_record
    ]

    if missing_fields:
        raise KeyError(
            f"Record kehilangan field: {missing_fields}. "
            f"Keys tersedia: {list(source_record.keys())}"
        )

    return {
        "image_id": image_id,
        "file_name": str(source_record["file_name"]),
        "width": int(source_record["width"]),
        "height": int(source_record["height"]),
        "boxes": [],
        "scores": [],
        "labels": [],
    }


print("✅ blank_record patched.")

✅ blank_record patched.


In [13]:
# CELL 10 — SIMPLE VALIDATION TUNING
tuning_rows = []

# Stage 1: thresholds and confidence type,
# using equal detector weights.
equal_weights = [1.0] * len(MODEL_NAMES)

best_stage_1 = None

for iou_thr in IOU_GRID:
    for skip_thr in SKIP_GRID:
        for conf_type in CONF_TYPES:
            fused = fuse_prediction_sets(
                validation_predictions,
                MODEL_NAMES,
                equal_weights,
                iou_thr,
                skip_thr,
                conf_type,
            )

            row = evaluate_coco(
                VAL_COCO,
                fused,
                COMBINATION_NAME,
            )

            row.update({
                "stage": "thresholds",
                "iou_thr": iou_thr,
                "skip_box_thr": skip_thr,
                "conf_type": conf_type,
                "weights": equal_weights,
            })

            tuning_rows.append(row)

            if (
                best_stage_1 is None
                or candidate_key(row)
                > candidate_key(best_stage_1)
            ):
                best_stage_1 = row

print("Best stage 1:")
print(best_stage_1)

# Stage 2: global pair weights,
# thresholds are locked from stage 1.
best_final = None

for first_weight in WEIGHT_GRID:
    for second_weight in WEIGHT_GRID:
        weights = [
            float(first_weight),
            float(second_weight),
        ]

        fused = fuse_prediction_sets(
            validation_predictions,
            MODEL_NAMES,
            weights,
            best_stage_1["iou_thr"],
            best_stage_1["skip_box_thr"],
            best_stage_1["conf_type"],
        )

        row = evaluate_coco(
            VAL_COCO,
            fused,
            COMBINATION_NAME,
        )

        row.update({
            "stage": "weights",
            "iou_thr": best_stage_1["iou_thr"],
            "skip_box_thr": best_stage_1[
                "skip_box_thr"
            ],
            "conf_type": best_stage_1[
                "conf_type"
            ],
            "weights": weights,
        })

        tuning_rows.append(row)

        if (
            best_final is None
            or candidate_key(row)
            > candidate_key(best_final)
        ):
            best_final = row

tuning_df = pd.DataFrame(tuning_rows)
tuning_df["weights"] = tuning_df[
    "weights"
].apply(json.dumps)

tuning_df.to_csv(
    RESULT_DIR / "validation_tuning_results.csv",
    index=False,
)

best_parameters = {
    "combination": COMBINATION_NAME,
    "models": MODEL_NAMES,
    "iou_thr": float(
        best_final["iou_thr"]
    ),
    "skip_box_thr": float(
        best_final["skip_box_thr"]
    ),
    "conf_type": best_final["conf_type"],
    "weights": [
        float(value)
        for value in best_final["weights"]
    ],
    "validation_mAP50_95": float(
        best_final["mAP50_95"]
    ),
    "validation_mAP50": float(
        best_final["mAP50"]
    ),
    "validation_mAP75": float(
        best_final["mAP75"]
    ),
    "validation_AP_coral": float(
        best_final["AP_coral"]
    ),
    "validation_AP_fish": float(
        best_final["AP_fish"]
    ),
}

save_json(
    best_parameters,
    RESULT_DIR / "best_wbf_parameters.json",
)

print("\n✅ Locked validation parameters:")
print(json.dumps(
    best_parameters,
    indent=2,
))

/usr/local/lib/python3.12/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [0.0, 0.0, 0.0, 0.7944725036621094].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.12/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [0.0, 0.5399921417236329, 0.0, 0.998837661743164].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.12/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [0.0, 0.23350035349527995, 0.0, 1.0].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.12/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [0.0, 0.0, 0.0, 0.8013902028401693].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.12/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: 

Best stage 1:
{'name': 'rtdetr_l_yolo11m', 'mAP50_95': 0.31709478454293794, 'mAP50': 0.49789384329133296, 'mAP75': 0.33734598355637835, 'AP_coral': 0.4299950006509719, 'AP_fish': 0.20419456843490402, 'worst_class_AP': 0.20419456843490402, 'detections': 32865, 'stage': 'thresholds', 'iou_thr': 0.6, 'skip_box_thr': 0.03, 'conf_type': 'max', 'weights': [1.0, 1.0]}

✅ Locked validation parameters:
{
  "combination": "rtdetr_l_yolo11m",
  "models": [
    "rtdetr_l",
    "yolo11m"
  ],
  "iou_thr": 0.6,
  "skip_box_thr": 0.03,
  "conf_type": "max",
  "weights": [
    0.75,
    0.75
  ],
  "validation_mAP50_95": 0.31709478454293794,
  "validation_mAP50": 0.49789384329133296,
  "validation_mAP75": 0.33734598355637835,
  "validation_AP_coral": 0.4299950006509719,
  "validation_AP_fish": 0.20419456843490402
}


In [14]:
# CELL 11 — LOCKED TEST EVALUATION
best_parameters = load_json(
    RESULT_DIR / "best_wbf_parameters.json"
)

fused_validation = fuse_prediction_sets(
    validation_predictions,
    MODEL_NAMES,
    best_parameters["weights"],
    best_parameters["iou_thr"],
    best_parameters["skip_box_thr"],
    best_parameters["conf_type"],
)

fused_test = fuse_prediction_sets(
    test_predictions,
    MODEL_NAMES,
    best_parameters["weights"],
    best_parameters["iou_thr"],
    best_parameters["skip_box_thr"],
    best_parameters["conf_type"],
)

save_json(
    fused_validation,
    RAW_DIR / "validation_wbf_fused.json",
)
save_json(
    fused_test,
    RAW_DIR / "test_wbf_fused.json",
)

ensemble_validation = evaluate_coco(
    VAL_COCO,
    fused_validation,
    COMBINATION_NAME,
)
ensemble_validation["split"] = "validation"
ensemble_validation["type"] = "ensemble"

ensemble_test = evaluate_coco(
    TEST_COCO,
    fused_test,
    COMBINATION_NAME,
)
ensemble_test["split"] = "test"
ensemble_test["type"] = "ensemble"

final_rows = []

for _, row in individual_df.iterrows():
    if row["split"] == "test":
        row_dict = row.to_dict()
        row_dict["type"] = "individual"
        final_rows.append(row_dict)

final_rows.append(ensemble_test)

final_test_df = (
    pd.DataFrame(final_rows)
    .sort_values(
        "mAP50_95",
        ascending=False,
    )
    .reset_index(drop=True)
)

final_test_df.to_csv(
    RESULT_DIR / "final_test_results.csv",
    index=False,
)

validation_summary_df = pd.DataFrame([
    ensemble_validation
])

validation_summary_df.to_csv(
    RESULT_DIR / "final_validation_ensemble.csv",
    index=False,
)

display(final_test_df)

print("\n✅ Final output:")
print(
    RESULT_DIR
    / "final_test_results.csv"
)
print(
    RESULT_DIR
    / "best_wbf_parameters.json"
)

,name,mAP50_95,mAP50,mAP75,AP_coral,AP_fish,worst_class_AP,detections,split,type
0,rtdetr_l_yolo11m,0.253685,0.440235,0.251595,0.292689,0.214680,0.214680,34608,test,ensemble
1,rtdetr_l,0.227832,0.407838,0.215314,0.258369,0.197296,0.197296,69300,test,individual
2,yolo11m,0.207680,0.350498,0.208917,0.242860,0.172501,0.172501,14484,test,individual



✅ Final output:
/content/drive/MyDrive/au-fish/dataset_v2/source_based/experiment_logs/simple_wbf_fullset_dedup_v1/rtdetr_l_yolo11m/results/final_test_results.csv
/content/drive/MyDrive/au-fish/dataset_v2/source_based/experiment_logs/simple_wbf_fullset_dedup_v1/rtdetr_l_yolo11m/results/best_wbf_parameters.json


## Output

Direktori hasil:

`/content/drive/MyDrive/au-fish/dataset_v2/source_based/experiment_logs/simple_wbf_fullset_dedup_v1/rtdetr_l_yolo11m/`

File utama:

- `results/final_test_results.csv`
- `results/best_wbf_parameters.json`
- `results/validation_tuning_results.csv`
- `results/individual_results.csv`
- `raw_predictions/test_wbf_fused.json`

Gunakan `final_test_results.csv` sebagai hasil akhir kombinasi ini.